# 01 — Generate the synthetic claims dataset

Runs all three stages of dataset construction (Section 3.2.1) on a Colab GPU and saves
everything to Google Drive:

| Stage | Module | Output (in Drive) |
|---|---|---|
| A. Structured claims | `src.data.generator` | `data/raw/<version>/base_claims.parquet`, `note_requests.jsonl` |
| B. LLM clinical notes | `src.data.note_synthesis` | `data/raw/<version>/notes.jsonl`, `boilerplate_notes.jsonl` |
| C. Injection, labels, splits | `src.data.build_dataset` | `data/processed/<version>/*.parquet`, `manifest.json` |

**Before running:** Runtime → Change runtime type → T4 GPU. Put a `.env` containing
`GITHUB_TOKEN=...` (a fine-grained, read-only token for `victorgpt0/claimguard`) at
`MyDrive/claimguard/.env`, or add it as a Colab secret; never paste it into a cell.

Stage B appends notes in batches, so if the runtime disconnects, rerun the notebook and it
resumes where it stopped.

In [3]:
DATASET_VERSION = 'v0.4-5k'   # bump when the generator, catalogue or parameters change
N_CLAIMS = 5000
SEED = 42
BRANCH = 'dev'

## 1. Bootstrap: Drive, repository, run record

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ROOT = '/content/drive/MyDrive/claimguard'
RAW = f'{ROOT}/data/raw/{DATASET_VERSION}'
PROCESSED = f'{ROOT}/data/processed/{DATASET_VERSION}'
ENV_LOGS = f'{ROOT}/artifacts/environment_logs'
for d in (RAW, PROCESSED, ENV_LOGS):
    os.makedirs(d, exist_ok=True)
print('commit', COMMIT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
commit 4a272d007fea2ff1f6ab25d642ee47650fccce10


In [5]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Python 3.13.15


## 2. Stage A: structured claims (CPU)

In [6]:
!pip install -q pyarrow pyyaml
!python -m src.data.generator --n-claims {N_CLAIMS} --seed {SEED} --out {RAW}

/content/claimguard/src/data/generator.py:271: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  claims[flag] = claims.get(flag, pd.Series(False, index=claims.index)).fillna(False).astype(bool)
/content/claimguard/src/data/generator.py:271: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  claims[flag] = claims.get(flag, pd.Series(False, index=claims.index)).fillna(False).astype(bool)
5000 claims, 60 providers -> /content/drive/MyDrive/claimguard/data/raw/v0.4-5k
  inflated caesarean claims: 81
  fictitious dependent claims: 83


## 3. Stage B: clinical notes (GPU)

Tries vLLM first (pinned to a release built for Colab's CUDA runtime). If the install or
start-up fails on the assigned GPU, set `BACKEND = 'transformers'`, which is slower
(about 1–2 h for 5,000 notes on a T4) but runs anywhere.

In [7]:
BACKEND = 'vllm'   # or 'transformers'
if BACKEND == 'vllm':
    !pip install -q vllm==0.19.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 1.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 5.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 59.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.3 MB/s eta 0:00:0000:

In [8]:
!python -m src.data.note_synthesis --backend {BACKEND} --seed {SEED} \
    --requests {RAW}/note_requests.jsonl --out {RAW}/notes.jsonl \
    --boilerplate-out {RAW}/boilerplate_notes.jsonl

I0000 00:00:1791754149.615388    9814 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 10-11 21:29:25 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 1024, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
config.json: 100% 841/841 [00:00<00:00, 4.87MB/s]
INFO 10-11 21:29:47 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 10-11 21:29:47 [model.py:1678] Using max model len 1024
INFO 10-11 21:29:48 [awq_marlin.py:249] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 10-11 21:29:48 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
model.safetensors.ind

### Inspect before building

Read a sample of notes yourself. Record what you checked, and anything you rejected, in
`docs/ai_use_log.md`: the notes are AI-generated research data.

In [9]:
import pandas as pd
from pathlib import Path   # a Path, so a missing file raises FileNotFoundError
notes = pd.read_json(Path(RAW, 'notes.jsonl'), lines=True)
boiler = pd.read_json(Path(RAW, 'boilerplate_notes.jsonl'), lines=True)
print(len(notes), 'notes;', notes['rejected_reason'].notna().sum(), 'rejected:')
print(notes['rejected_reason'].value_counts())
print(len(boiler), 'boilerplate;', boiler['rejected_reason'].notna().sum(), 'rejected')
requests = pd.read_json(Path(RAW, 'note_requests.jsonl'), lines=True)
sample = notes.merge(requests, on='claim_id').sample(8, random_state=SEED)
for row in sample.itertuples():
    print(f'\n[{row.icd11_code}] {row.diagnosis} | {row.patient_age}{row.patient_sex[0]} {row.package}')
    print(row.clinical_notes)

/tmp/ipykernel_5955/3052199624.py:2: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  notes = pd.read_json(f'{RAW}/notes.jsonl', lines=True)


ValueError: Expected object or value

In [ ]:
print('Accepted boilerplate:')
for n in boiler.loc[boiler['rejected_reason'].isna(), 'clinical_notes'].sample(10, random_state=SEED):
    print(' -', n)

## 4. Stage C: inject fraud, label, split

In [ ]:
!python -m src.data.build_dataset --raw {RAW} --out {PROCESSED} --seed {SEED}

In [ ]:
import json, shutil, datetime
manifest = json.load(open(f'{PROCESSED}/manifest.json'))
manifest['colab_commit'] = COMMIT
json.dump(manifest, open(f'{PROCESSED}/manifest.json', 'w'), indent=2)
stamp = datetime.date.today().isoformat()
!pip freeze > {ENV_LOGS}/requirements_frozen_{stamp}_dataset.txt
pd.DataFrame(manifest['splits']).T[['claims', 'fraud', 'inconsistent', 'text_noise']]